# Deploying a Generative Program as an API with `m serve`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/DeployGenerativeProgram/DeployGenerativeProgram.ipynb)

A generative program becomes useful when other software can call it. In this recipe you turn a Mellea program into a REST API with **one command**, `m serve`, and call it like any OpenAI model.

**What you will build:** a PII redactor API. Send it text, and it returns the text with names, email addresses, phone numbers and social security numbers replaced by placeholders. It checks its own output and repairs mistakes before answering.

**What you will be able to do afterward:**

- Serve any Mellea program as an OpenAI-compatible API
- Let clients add requirements and model options per request
- Package the service in a container

**Prerequisites:** Python 3.11 or 3.12, and an IBM watsonx.ai instance. This recipe uses the IBM® [Granite](https://www.ibm.com/granite) model on watsonx.ai.

## Step 1. Install the libraries

The `cli` and `server` extras add the `m` command and the web server behind it.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    "mellea[litellm,cli,server]>=0.8.0" openai requests
! echo "::endgroup::"

## Step 2. Load your watsonx.ai credentials

You need `WATSONX_URL`, `WATSONX_APIKEY` and `WATSONX_PROJECT_ID`. See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) if you don't have them yet.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

get_env_var("WATSONX_APIKEY")
get_env_var("WATSONX_PROJECT_ID")
get_env_var("WATSONX_URL")

## Step 3. Write the program

`m serve` serves a Python file that defines a `serve()` function. It calls that function for every request with three arguments:

| Argument | What it contains |
| --- | --- |
| `input` | The chat messages sent by the client |
| `requirements` | Extra requirements sent by the client |
| `model_options` | Options such as `temperature`, `max_tokens` and `seed` |


Our `serve()` is an ordinary Mellea program. 

* A Python check, `no_pii_left()`, catches leftover email addresses, phone numbers and SSNs. 

* When it fails, `RepairTemplateStrategy` sends its message back to the model, which tries again. 

* If the output still fails after three attempts, the program raises an error rather than return text that might leak PII.

In [ ]:
%%writefile pii_redactor.py
"""PII redactor: a validated Mellea program, ready for `m serve`."""

import os
import re

import mellea
from mellea.core import ModelOutputThunk
from mellea.serve import ChatMessage
from mellea.stdlib.requirements import req, simple_validate
from mellea.stdlib.sampling import RepairTemplateStrategy

MODEL_ID = os.getenv("MELLEA_MODEL_ID", "watsonx/ibm/granite-4-h-small")

PII_PATTERNS = {
    "[EMAIL]": r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "[PHONE]": r"(?<!\w)(?:\+?1[ .-]?)?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b",
    "[SSN]": r"\b\d{3}-\d{2}-\d{4}\b",
}


def no_pii_left(output: str) -> tuple[bool, str]:
    """Python check: fail, and say exactly what was missed, if PII remains."""
    missed = [
        f"{value} -> {tag}"
        for tag, pattern in PII_PATTERNS.items()
        for value in re.findall(pattern, output)
    ]
    if missed:
        return False, "Still visible, replace them: " + "; ".join(missed)
    return True, "No email address, phone number or SSN remains."


def serve(
    input: list[ChatMessage],
    requirements: list[str] | None = None,
    model_options: dict | None = None,
) -> ModelOutputThunk:
    """Called by `m serve` for every request."""
    m = mellea.start_session(backend_name="litellm", model_id=MODEL_ID)
    result = m.instruct(
        "Redact the personal information in the text below. Replace names with "
        "[NAME], email addresses with [EMAIL], phone numbers with [PHONE] and "
        "social security numbers with [SSN]. Replace only the values, keep "
        "everything else unchanged, and return only the redacted text."
        "\n\nText:\n{{text}}",
        user_variables={"text": input[-1].get_text_content()},
        requirements=[
            req(
                "Replace every email address, phone number and SSN",
                validation_fn=simple_validate(no_pii_left),
            ),
            req("Replace every person's name with [NAME]"),
            *[r for r in requirements or [] if r],  # extra rules from the client
        ],
        strategy=RepairTemplateStrategy(loop_budget=3),
        return_sampling_results=True,
        model_options=model_options,
    )
    if not result.success:
        # Fail closed: never return text that did not pass validation.
        raise RuntimeError("Redaction failed validation")
    return result.result

## Step 4. Try it locally

Before deploying, call `serve()` directly, exactly as the server will. `%run` loads the latest version of the file, so you can edit Step 3 and try again.

In [ ]:
%run pii_redactor.py

ticket = (
    "Hi team, Maria Lopez (maria.lopez@example.com, +1 415-555-0132) called about "
    "order 88231. Her SSN is 123-45-6789. Call her back before 5pm."
)

print(serve([ChatMessage(role="user", content=ticket)]).value)

If the log above shows `FAILED` before the result, you just watched the repair loop work: an attempt missed something, and the next one fixed it. This is the feedback the model receives in that case:

In [ ]:
print(no_pii_left("Call Maria at 415-555-0132."))

## Step 5. Serve it with one command

In a terminal, this is all it takes:

```bash
m serve pii_redactor.py
```

In a notebook, we start the same command in the background and wait until its health check answers. We bind it to `127.0.0.1` so that only this machine can reach it.

In [ ]:
import shutil
import subprocess
import sys
import time
from pathlib import Path
import requests

BASE_URL = "http://127.0.0.1:8081"
m_cli = shutil.which("m") or str(Path(sys.executable).with_name("m"))

# Start server and capture output to diagnose potential issues
server = subprocess.Popen(
    [m_cli, "serve", "pii_redactor.py", "--host", "127.0.0.1", "--port", "8081"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait and check health
for i in range(15):
    try:
        res = requests.get(f"{BASE_URL}/health")
        if res.ok:
            print("API is up! Response:", res.json())
            break
    except Exception as e:
        time.sleep(1)
else:
    print("Server failed to start. Diagnostics:")
    print("Exit code:", server.poll())
    stdout, stderr = server.communicate(timeout=2)
    print("--- STDOUT ---\n", stdout)
    print("--- STDERR ---\n", stderr)

## Step 6. Call your API

The API speaks the OpenAI chat completions format, so the official `openai` client works by changing only the base URL.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="pii-redactor",
    messages=[{"role": "user", "content": ticket}],
    temperature=0,
)
print(response.choices[0].message.content)

Clients can also tighten the rules for a single request. Here we add a requirement to redact company names. `temperature` and the other model options are passed to `serve()` as `model_options`.

In [ ]:
response = client.chat.completions.create(
    model="pii-redactor",
    messages=[{"role": "user", "content": "Tom Baker from ACME Corp approved invoice 4471."}],
    temperature=0,
    extra_body={"requirements": ["Replace every company name with [COMPANY]"]},
)
print(response.choices[0].message.content)

**Your generative program is now an API.** Any language or tool that can send HTTP can use it, for example:

```bash
curl http://127.0.0.1:8081/v1/chat/completions -H "Content-Type: application/json" \
  -d '{"model": "pii-redactor", "messages": [{"role": "user", "content": "Call Maria at 415-555-0132."}]}'
```

## Step 7. Ship it

To deploy the service, put the program in a container. This `Dockerfile` is all you need:

```dockerfile
FROM python:3.12-slim
RUN pip install --no-cache-dir "mellea[litellm,cli,server]==0.8.0"
WORKDIR /app
COPY pii_redactor.py .
CMD ["m", "serve", "pii_redactor.py", "--host", "0.0.0.0", "--port", "8081"]
```

```bash
docker build -t pii-redactor .
docker run -p 8081:8081 -e WATSONX_URL -e WATSONX_APIKEY -e WATSONX_PROJECT_ID pii-redactor
```

Before real users depend on it:

- **Protect it.** `m serve` has no authentication, so put it behind an API gateway or reverse proxy.
- **Scale it.** Run more containers behind a load balancer, and point their health probes at `/health`.
- **Pin it.** Pin the Mellea version and model, and re-run Step 4 before upgrading either.

## Step 8. Stop the server

In [ ]:
server.terminate()
server.wait()
print("Server stopped.")

## Notebook Takeaways

- `m serve your_program.py` turns a `serve()` function into an OpenAI-compatible API, so existing OpenAI clients can call your generative program.
- Clients can add requirements and model options per request, while the program keeps the rules that matter.
- The same file runs in a notebook, behind `m serve`, and in a container.